In [1]:

import os
import glob
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:

import glob
import pandas as pd

paths = glob.glob("/kaggle/input/**/train.csv", recursive=True)
test_paths = glob.glob("/kaggle/input/**/test.csv", recursive=True)
sample_paths = glob.glob(
    "/kaggle/input/**/sample_submission.csv",
    recursive=True
)

print("Train files:", paths)
print("Test files:", test_paths)
print("Sample files:", sample_paths)

if not paths or not test_paths or not sample_paths:
    print("ERROR: Competition dataset attach nahi hai.")
else:
    train = pd.read_csv(paths[0])
    test = pd.read_csv(test_paths[0])
    sample = pd.read_csv(sample_paths[0])

    print("Train loaded:", train.shape)
    print("Test loaded:", test.shape)
    print("Dataset successfully loaded!")


Train files: ['/kaggle/input/competitions/llm-classification-finetuning/train.csv']
Test files: ['/kaggle/input/competitions/llm-classification-finetuning/test.csv']
Sample files: ['/kaggle/input/competitions/llm-classification-finetuning/sample_submission.csv']
Train loaded: (57477, 9)
Test loaded: (3, 4)
Dataset successfully loaded!


In [3]:

target_cols = [
    "winner_model_a",
    "winner_model_b",
    "winner_tie"
]

print("Missing values:")
print(train[
    ["prompt", "response_a", "response_b"]
].isnull().sum())

print("\nTarget distribution:")
print(train[target_cols].sum())

print("\nSubmission columns:")
print(sample.columns.tolist())


Missing values:
prompt        0
response_a    0
response_b    0
dtype: int64

Target distribution:
winner_model_a    20064
winner_model_b    19652
winner_tie        17761
dtype: int64

Submission columns:
['id', 'winner_model_a', 'winner_model_b', 'winner_tie']


In [4]:

text_cols = ["prompt", "response_a", "response_b"]

def make_text(df):
    return (
        "PROMPT " + df["prompt"].fillna("").astype(str)
        + " RESPONSE_A "
        + df["response_a"].fillna("").astype(str)
        + " RESPONSE_B "
        + df["response_b"].fillna("").astype(str)
    )

X_text = make_text(train)
X_test_text = make_text(test)

y = train[
    ["winner_model_a", "winner_model_b", "winner_tie"]
].to_numpy().argmax(axis=1)

print("X_text created:", len(X_text))
print("X_test_text created:", len(X_test_text))
print("Labels created:", len(y))


X_text created: 57477
X_test_text created: 3
Labels created: 57477


In [5]:

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
import numpy as np

X_train, X_valid, y_train, y_valid = train_test_split(
    X_text,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 1),
    min_df=3,
    dtype=np.float32
)

X_train_vec = vectorizer.fit_transform(X_train)
X_valid_vec = vectorizer.transform(X_valid)

model = LogisticRegression(
    max_iter=100,
    solver="liblinear"
)

model.fit(X_train_vec, y_train)

valid_probs = model.predict_proba(X_valid_vec)

print(
    "Validation log loss:",
    log_loss(y_valid, valid_probs, labels=[0, 1, 2])
)


Validation log loss: 1.107403449441999


In [6]:

X_full_vec = vectorizer.fit_transform(X_text)
X_test_vec = vectorizer.transform(X_test_text)

model.fit(X_full_vec, y)

predictions = model.predict_proba(X_test_vec)

target_cols = [
    "winner_model_a",
    "winner_model_b",
    "winner_tie"
]

submission = pd.DataFrame(
    predictions,
    columns=target_cols
)

submission.insert(0, "id", test["id"].values)


submission = submission[sample.columns]


submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("Submission created successfully!")
print("Submission shape:", submission.shape)
display(submission.head())


Submission created successfully!
Submission shape: (3, 4)


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.227555,0.339314,0.433131
1,211333,0.310019,0.373350,0.316631
2,1233961,0.327402,0.476868,0.195730


In [7]:

import os
import numpy as np
import pandas as pd

file_path = "/kaggle/working/submission.csv"

assert os.path.exists(file_path), "submission.csv file not found!"

submission = pd.read_csv(file_path)

target_cols = [
    "winner_model_a",
    "winner_model_b",
    "winner_tie"
]

assert list(submission.columns) == list(sample.columns), \
    "Column names or order do not match!"

assert len(submission) == len(test), \
    "Submission row count does not match test data!"

assert submission[target_cols].notna().all().all(), \
    "Missing predictions found!"

probs = submission[target_cols].to_numpy()

assert np.isfinite(probs).all(), \
    "Invalid numeric values found!"

assert (probs >= 0).all() and (probs <= 1).all(), \
    "Probabilities must be between 0 and 1!"

assert np.allclose(probs.sum(axis=1), 1.0, atol=1e-5), \
    "Probabilities must sum to 1!"

assert submission["id"].equals(sample["id"]), \
    "IDs do not match sample submission!"

print("All submission checks passed!")
print("Total rows:", len(submission))
print("Total columns:", len(submission.columns))
display(submission.head())


All submission checks passed!
Total rows: 3
Total columns: 4


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.227555,0.339314,0.433131
1,211333,0.310019,0.373350,0.316631
2,1233961,0.327402,0.476868,0.195730
